# 04 · Two-tower finder

The trending finder gives everyone the same 500 articles. The two-tower finder builds a cart **for you**:

- **Item tower:** turns each article into a list of numbers (a *vector*) from its text.
- **User tower:** turns *you* into a vector from the articles you clicked before.
- Articles whose vectors are closest to yours are the ones you'd probably like.

## Step 1: Article text → numbers

We use **MiniLM**, a small pre-trained language model, *frozen* (we don't retrain it). It reads each article's category, subcategory, title and abstract and outputs 384 numbers. Because it reads text, it works for brand-new articles too, which matters because 36% of validation clicks go to articles never seen in training.

This runs once (~1–2 minutes) and is saved to `artifacts/news_embeddings.npz`. Later runs just load the file.

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd

from src.data import load_news
from src.finder import EMBEDDINGS_PATH, article_text, embed_articles, load_embeddings, save_embeddings

In [ ]:
news = load_news()

if not EMBEDDINGS_PATH.exists():
    save_embeddings(news.news_id, embed_articles(news))

news_ids, vectors = load_embeddings()
assert (news_ids == news.news_id.values).all()
print(f"{vectors.shape[0]:,} articles × {vectors.shape[1]} numbers each")
print("Example of the text MiniLM reads:\n ", article_text(news).iloc[0])

## Sanity check: do similar articles get similar numbers?

If MiniLM works, an article's closest neighbors (by vector) should be about the same thing. The similarity score runs from 0 (unrelated) to 1 (identical).

In [ ]:
def neighbors(news_id, n=5):
    i = np.flatnonzero(news_ids == news_id)[0]
    similarity = vectors @ vectors[i]
    top = np.argsort(-similarity)[: n + 1]
    return news.iloc[top][["bucket", "subcategory", "title"]].assign(similarity=similarity[top].round(2))

def find(word, subcategory):
    match = news[(news.subcategory == subcategory) & news.title.str.contains(word, case=False)]
    return match.news_id.iloc[0]

In [ ]:
neighbors(find("Patriots", "football_nfl"))

In [ ]:
neighbors(find("recipe", "recipes"))

In [ ]:
neighbors(find("retire", "finance-retirement"))

## Do neighbors usually share a bucket?

Across 2,000 random articles: how often are the 5 nearest neighbors in the same bucket? Random would be about 20–30%. A high number means the vectors capture topic well, which the arranger and the bucket quotas rely on.

In [ ]:
rng = np.random.default_rng(0)
sample = rng.choice(len(news), 2000, replace=False)
buckets = news.bucket.to_numpy(dtype=object)

similarity = vectors[sample] @ vectors.T
similarity[np.arange(len(sample)), sample] = -1  # ignore the article itself
top5 = np.argsort(-similarity, axis=1)[:, :5]

has_bucket = pd.notna(buckets[sample])
same = (buckets[top5] == buckets[sample][:, None])[has_bucket]
print(f"Nearest 5 neighbors in the same bucket: {100 * same.mean():.0f}%")
pd.Series(same.mean(axis=1), index=buckets[sample][has_bucket]).groupby(level=0).mean().mul(100).round(0).rename("same_bucket_%")

## Takeaways (Step 1)

- **MiniLM understands topics.** A Patriots–Ravens preview finds other Patriots–Ravens stories (similarity 0.81–0.85), a gravy recipe finds other Thanksgiving recipes, and an "retire in an RV" article finds other RV-retirement pieces.
- **82% of nearest neighbors share a bucket.** Sports is almost perfectly separated (96%). Entertain is the fuzziest (60%), which makes sense: celebrity stories overlap with news and lifestyle.
- **Cross-bucket neighbors can be sensible**, e.g. "retire in an RV" (Learn) sits next to "26 Little-Known Facts About RVs" (Lifestyle). Buckets are categories, but meaning doesn't always follow category lines.

Next: **Step 2**, train the user and item towers on Nov 9–13 clicks.